# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulwahab-git/week-01-Assignment/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Primary Research Question:
Can tree-based ensemble learning models accurately predict significant long-term organic search traffic drops within an enterprise website architecture before visibility entirely evaporates?

Supported Decision Boundary:
This pipeline acts as an automated directional decision-support tool for content teams. It surfaces a prioritized operational queue of high-risk pages, allowing managers to allocate engineering hours to optimize volatile content assets before permanent search position drops occur.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify question tracking parameters
research_target_metric = "gsc_impressions"
prediction_goal = "is_decline_target"

print(f"Target metric identified: {research_target_metric}")
print(f"Prediction optimization node calibrated for: {prediction_goal}")

Target metric identified: gsc_impressions
Prediction optimization node calibrated for: is_decline_target


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

This study uses the FlyRank ML Internship dataset, accessed through the public Hugging Face dataset release FlyRank/internship-warehouse. The analysis focuses on the fact_content_daily_performance data, which contains daily content-performance observations used to study changes in organic search visibility.

The primary target metric is Google Search Console impressions (gsc_impressions). The analysis uses the available 2026 monthly Parquet partitions from the warehouse and establishes the maximum available report_date programmatically before downstream analysis. This approach avoids hard-coding the production timeline and provides a reproducible boundary for the analysis.

The dataset was queried through DuckDB directly against the Parquet files rather than loading the complete warehouse into memory. A 10 GB DuckDB memory limit was applied to keep the analysis reproducible within a constrained computational environment.

For public presentation, the study reports only aggregate analytical findings and methodology. Client names, private URLs, private search queries, and other identifying or sensitive information are excluded. The analysis is intended to demonstrate a generalizable modeling workflow rather than expose individual client or page-level confidential information.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import pandas as pd
from google.colab import userdata

print("Establishing secure connection to the backend data warehouse...")

# Authenticate with the data layer via user keys
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')")

# Enforce a strict RAM limit on the query execution environment to prevent system memory crashes
con.execute("SET memory_limit='10GB'")
print("DuckDB authenticated and engine memory caps successfully configured!")

# Structure remote parquet paths
DATA_WAREHOUSE_URL = "hf://datasets/FlyRank/internship-warehouse"
fact_daily_path = f"read_parquet('{DATA_WAREHOUSE_URL}/fact_content_daily_performance/month=2026-0*/*.parquet')"

# Extract maximum baseline dates in a quick scalar pass to conserve runtime memory
max_date_query = f"SELECT MAX(report_date) FROM {fact_daily_path}"
target_max_date = con.execute(max_date_query).fetchone()[0]

print(f"Verified Production Timeline Horizon Bound: {target_max_date}")

Establishing secure connection to the backend data warehouse...
Paste your Hugging Face READ token: ··········
DuckDB authenticated and engine memory caps successfully configured!


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Verified Production Timeline Horizon Bound: 2026-06-30


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Core Assumptions

The analysis assumes that structural search-performance signals and historical site activity contain information about future performance stability. In particular, historical organic search impressions and the number of active reporting days are treated as operational indicators that may help distinguish relatively stable content assets from assets experiencing significant declines.

### Features

Two features are used for the initial prediction matrix:

* `impressions_prior` — historical organic search impressions accumulated before the most recent 90-day window.
* `active_days_count` — the number of distinct reporting days observed for the content node.

The target-construction variables `impressions_last_90d` and `impressions_prior` are used to calculate the decline score. They are not treated as independent predictive features in the current feature matrix beyond the explicitly selected `impressions_prior` feature.

### Label Definition

The study converts the continuous change in search presence into a binary operational label. The decline score is defined as:

`action_score = impressions_last_90d / (impressions_prior + 1)`

A content node is assigned the positive decline label when:

`action_score < 0.8`

Therefore, the model is designed to identify content nodes whose recent 90-day search-impression volume is below 80% of the corresponding prior-period baseline.

The `+1` denominator adjustment prevents division-by-zero errors for content nodes with no prior-period impressions.

### Validation Strategy

To reduce the risk of client-specific memorization and data leakage, the evaluation uses a client-grouped holdout. Approximately 80% of the unique client groups are assigned to the training partition, while the remaining client groups form the evaluation partition.

This means that the evaluation set contains client groups not used to train the model. The resulting score is therefore intended to provide a more realistic estimate of how the approach generalizes across previously unseen enterprise domains rather than how well it performs on pages from clients already observed during training.

### Leakage Considerations

The validation design is intended to prevent information from the same client group appearing in both the training and evaluation partitions. Features are constructed from historical performance windows, while the binary decline label represents the defined comparison between recent and prior search-impression periods.

The analysis therefore treats the client-grouped split as an important safeguard against client-specific patterns artificially inflating evaluation performance. Any remaining temporal or feature-construction limitations are treated as limitations of the current experimental design rather than evidence of causal prediction.

### Baseline

The model should be compared against a simple baseline evaluated on the **same client-grouped holdout**. This ensures that any reported improvement reflects performance beyond a straightforward reference strategy rather than a difference caused by using different evaluation samples.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Beginning memory-safe data aggregation query...")

optimized_aggregation_query = f"""
SELECT
    f.client_hash_id,
    f.content_hash_id,
    SUM(CASE WHEN f.report_date > CAST('{target_max_date}' AS DATE) - INTERVAL 90 DAY THEN f.gsc_impressions ELSE 0 END) AS impressions_last_90d,
    SUM(CASE WHEN f.report_date <= CAST('{target_max_date}' AS DATE) - INTERVAL 90 DAY THEN f.gsc_impressions ELSE 0 END) AS impressions_prior,
    COUNT(DISTINCT f.report_date) AS active_days_count
FROM {fact_daily_path} f
GROUP BY f.client_hash_id, f.content_hash_id
"""

# Extract structural data frame matrix
master_features_df = con.sql(optimized_aggregation_query).df()

# Calculate targeted evaluation parameters
master_features_df["action_score"] = master_features_df["impressions_last_90d"] / (master_features_df["impressions_prior"] + 1)
master_features_df["is_decline_target"] = (master_features_df["action_score"] < 0.8).astype(int)

print(f"Data layer parsing complete. Processed record volume: {len(master_features_df):,} lines.")

# Execute strict client-grouped partitioning split boundaries
unique_clients = master_features_df["client_hash_id"].unique()
split_barrier = int(len(unique_clients) * 0.8)
train_clients = unique_clients[:split_barrier]

train_mask = master_features_df["client_hash_id"].isin(train_clients)
training_set = master_features_df[train_mask]
evaluation_set = master_features_df[~train_mask]

feature_columns = ["impressions_prior", "active_days_count"]
X_train, y_train = training_set[feature_columns], training_set["is_decline_target"]
X_eval, y_eval = evaluation_set[feature_columns], evaluation_set["is_decline_target"]

print(f"Grouped Training Matrix: {len(X_train):,} rows | Grouped Validation Matrix: {len(X_eval):,} rows")

Beginning memory-safe data aggregation query...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data layer parsing complete. Processed record volume: 427,292 lines.
Grouped Training Matrix: 396,970 rows | Grouped Validation Matrix: 30,322 rows


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*


The Random Forest classifier was evaluated against a simple majority-class baseline using the same client-grouped evaluation partition. This comparison provides a common reference point for determining whether the learned model extracts useful predictive structure beyond simply assigning every evaluation observation to the most common training class.

The Random Forest model used 100 trees with a maximum tree depth of 8 and a fixed random seed of 42 to make the experiment reproducible. Performance was assessed using accuracy, precision, recall, and F1-score.

Because the positive class represents content nodes meeting the defined decline criterion, precision measures how often predicted high-risk nodes actually belong to the decline class, while recall measures how many of the observed decline cases are identified by the model. F1-score provides a combined measure of precision and recall.

The results below report both approaches on the same held-out client groups. The comparison is intended as a directional evaluation of the model's usefulness for prioritizing potentially declining content, rather than as evidence of causal prediction or guaranteed future traffic recovery.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Training production ensemble random forest classification nodes...")

predictive_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42
)

predictive_model.fit(X_train, y_train)

# Machine-learning predictions
ml_predictions = predictive_model.predict(X_eval)

# ---------------------------------------------------------
# Honest baseline: majority class from TRAINING data only
# ---------------------------------------------------------
majority_class = y_train.mode()[0]
baseline_predictions = [majority_class] * len(y_eval)

# Compile comparison
results_comparison_df = pd.DataFrame({
    "Performance Evaluation Parameter": [
        "Pipeline Accuracy",
        "Target Precision",
        "Target Recall (Sensitivity)",
        "Balanced F1-Score"
    ],

    "Majority-Class Baseline": [
        f"{accuracy_score(y_eval, baseline_predictions):.2%}",
        f"{precision_score(y_eval, baseline_predictions, zero_division=0):.2%}",
        f"{recall_score(y_eval, baseline_predictions, zero_division=0):.2%}",
        f"{f1_score(y_eval, baseline_predictions, zero_division=0):.2%}"
    ],

    "Machine Learning Model": [
        f"{accuracy_score(y_eval, ml_predictions):.2%}",
        f"{precision_score(y_eval, ml_predictions, zero_division=0):.2%}",
        f"{recall_score(y_eval, ml_predictions):.2%}",
        f"{f1_score(y_eval, ml_predictions, zero_division=0):.2%}"
    ]
})

print("\n========================================================================")
print("             PRODUCTION PERFORMANCE MATRICES REPORTING TABLE")
print("========================================================================")
print(results_comparison_df.to_string(index=False))
print("========================================================================")

Training production ensemble random forest classification nodes...

             PRODUCTION PERFORMANCE MATRICES REPORTING TABLE
Performance Evaluation Parameter Majority-Class Baseline Machine Learning Model
               Pipeline Accuracy                  28.17%                 66.54%
                Target Precision                  28.17%                 42.85%
     Target Recall (Sensitivity)                 100.00%                 56.22%
               Balanced F1-Score                  43.96%                 48.63%


## 5. Limitations

*What this work cannot claim.*

This model operates on structural search-footprint behavior and historical performance trends. It cannot reliably account for macro-level system changes, including broad search-engine infrastructure or algorithm updates, intentional directory migrations, or seasonal and cyclical changes in search demand. Volatile or newly tracked content nodes with little or no historical baseline impressions are also more prone to false hazard warnings because the model has limited historical evidence from which to establish a meaningful baseline. Such cases should therefore be separated from the automated prioritization queue and reviewed manually by a qualified specialist.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Isolate high-risk edge cases:
# content nodes with a decline label but zero prior tracking volume
edge_case_mask = (
    (evaluation_set["is_decline_target"] == 1)
    & (evaluation_set["impressions_prior"] == 0)
)

print(
    f"Total volatile system boundaries flagged for manual rule exclusion: "
    f"{edge_case_mask.sum():,}"
)

Total volatile system boundaries flagged for manual rule exclusion: 8,533


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Rather than limiting the operational output to binary classifications, the action playbook uses the model's continuous probability estimates to rank content nodes by relative decline risk. This produces a dynamically ordered optimization backlog, allowing content teams to focus attention first on assets receiving the strongest model-based risk signal.

Three operational tiers are used:

CRITICAL_URGENT_SALVAGE — risk score ≥ 0.80. These assets receive the highest operational priority and should be reviewed urgently for potential search-footprint deterioration.
OPTIMIZE_CONTENT_FOOTPRINT — risk score ≥ 0.50 and < 0.80. These assets represent elevated-risk candidates for content and technical review.
MONITOR_STABLE — risk score < 0.50. These assets remain in the monitoring pool rather than receiving immediate optimization priority.

Human sign-off is required for assets with a risk score of 0.50 or higher. The ranking is intended as a decision-support mechanism: the score determines prioritization, while a human specialist determines whether intervention is appropriate.

The resulting queue is sorted from highest to lowest risk score and exported as final_action_playbook.csv for inspection and reuse in the research artifacts.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# Compute continuous risk probabilities for operational prioritization
risk_probabilities = predictive_model.predict_proba(X_train)[:, 1]

# Construct the operational action backlog
action_playbook_df = training_set[
    ["client_hash_id", "content_hash_id", "impressions_last_90d"]
].copy()

action_playbook_df["risk_score"] = risk_probabilities

# Assign operational risk tiers
action_playbook_df["operational_action"] = "MONITOR_STABLE"

action_playbook_df.loc[
    action_playbook_df["risk_score"] >= 0.5,
    "operational_action"
] = "OPTIMIZE_CONTENT_FOOTPRINT"

action_playbook_df.loc[
    action_playbook_df["risk_score"] >= 0.8,
    "operational_action"
] = "CRITICAL_URGENT_SALVAGE"

# Human review for elevated-risk assets
action_playbook_df["requires_human_signoff"] = (
    action_playbook_df["risk_score"] >= 0.5
)

# Rank from highest to lowest risk
ranked_playbook = (
    action_playbook_df
    .sort_values(by="risk_score", ascending=False)
    .reset_index(drop=True)
)

# Export
output_directory = "../outputs"

if not os.path.exists(output_directory):
    os.makedirs(output_directory)

ranked_playbook.to_csv(
    f"{output_directory}/final_action_playbook.csv",
    index=False
)

print(
    f"Successfully compiled and written "
    f"{len(ranked_playbook):,} records to outputs sandbox directory."
)

Successfully compiled and written 396,970 records to outputs sandbox directory.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The paper embeds the core quantitative artifacts generated by the analysis. These artifacts provide a compact view of model performance and the resulting operational prioritization queue.

The first artifact compares the machine-learning model against the selected baseline using the same held-out evaluation partition. The second artifact presents the highest-risk content nodes from the action playbook, including their model-derived risk scores and corresponding operational action tiers.

Together, these artifacts connect the validation results to the practical decision-support objective of the study: identifying content assets that may warrant earlier human review and optimization attention.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the core summary artifacts for submission validation

print("=== Embedding Artifact 1: Model Performance vs Baseline ===")
print(results_comparison_df.to_string(index=False))

print("\n=== Embedding Artifact 2: Top Urgent Optimization Backlog ===")
print(
    ranked_playbook[
        ["client_hash_id", "risk_score", "operational_action"]
    ]
    .head(5)
    .to_string(index=False)
)

=== Embedding Artifact 1: Model Performance vs Baseline ===
Performance Evaluation Parameter Majority-Class Baseline Machine Learning Model
               Pipeline Accuracy                  28.17%                 66.54%
                Target Precision                  28.17%                 42.85%
     Target Recall (Sensitivity)                 100.00%                 56.22%
               Balanced F1-Score                  43.96%                 48.63%

=== Embedding Artifact 2: Top Urgent Optimization Backlog ===
         client_hash_id  risk_score      operational_action
client_2e65897d94f60220         1.0 CRITICAL_URGENT_SALVAGE
client_2e65897d94f60220         1.0 CRITICAL_URGENT_SALVAGE
client_2e65897d94f60220         1.0 CRITICAL_URGENT_SALVAGE
client_2e65897d94f60220         1.0 CRITICAL_URGENT_SALVAGE
client_2e65897d94f60220         1.0 CRITICAL_URGENT_SALVAGE


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.